In [24]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append('..')

import utilities.functions as functions

from utilities.functions import (
    load_data,
    check_key_uniqueness,
    merge_df,
    load_orders,
    process_orders_pandas
)

In [25]:
from pathlib import Path
BASE_PATH = Path("/Users/maceli/ifood_cs/dados") 



Load the data --- se necessario salvar em stage - neste momento o estara comentado

In [26]:
URL_CONSUMER = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/consumer.csv.gz"
df_consumer = load_data(URL_CONSUMER)[["customer_id", "active","created_at"]]

In [27]:
URL_RESTAURANT ="https://data-architect-test-source.s3-sa-east-1.amazonaws.com/restaurant.csv.gz"
df_restaurant= load_data(URL_RESTAURANT)

In [28]:
ab_test_url = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/ab_test_ref.tar.gz"
df_ab = load_data(ab_test_url)

In [29]:
#save_parquet(df_consumer, "stage", "df_consumer.parquet")
#save_parquet(df_restaurant, "stage", "df_restaurant.parquet")
#save_parquet(df_consumer, "stage", "df_consumer.parquet")

Bronze layer - verify duplicates e nulo. e se necessario remover

In [30]:
check_key_uniqueness(df_consumer, ["customer_id","active","created_at"])

✅ Colunas ['customer_id', 'active', 'created_at'] são NOT NULL e UNIQUE.


(True, None, None, None)

In [31]:
check_key_uniqueness(df_restaurant, ["id"])

✅ Colunas ['id'] são NOT NULL e UNIQUE.


(True, None, None, None)

In [32]:
check_key_uniqueness(df_ab, ["customer_id","is_target"])
print(df_ab[df_ab["customer_id"].isna()])


❌ Colunas ['customer_id', 'is_target'] contêm valores nulos.

Soma de nulos por coluna:
customer_id    1
is_target      0
dtype: int64

Índices com nulos:
[81149]
      customer_id is_target
81149         NaN    target


In [33]:
df_ab_np = df_ab[~df_ab["customer_id"].isna()]

Antes de carregar a base ordens sera definido o publico todal, e da base de ordem serao filtrados somentes os clientes elegiceis

In [34]:
#import os

# Listar todos os arquivos de um diretório
#arquivos = os.listdir(BASE_PATH / "gold" )
#print("Arquivos no diretório:")
#for arquivo in arquivos:
    #print(f"  - {arquivo}")

In [35]:
#df = pd.read_json(BASE_PATH / "gold" / 'amostra_aleatoria.json')

In [36]:
amostra_aleatoria = df_ab.sample(n=100000, random_state=42)['customer_id']

id=['fffe7b38b14ac2fca8906500fbecf87f5f8470179ed7ffa974892a3a15650604',
       'fffd6a19e4affba4589945ba2fe76804f25ad9301c0d3011219766b51106c2fe',
       'fffad85994233d99370bfb55ff196c7e82af11dd9825b121fa5f5563c2666c2a',
       '00086dd0b93a9c96d2d13e1b245bb82abbea957306b193be39375bdd853811f9',
       '00070a129efd4c5ffe3dfbc2ca704a7f891fd8b0ab4159930b813c541194c0cc',
       '0000c21984ae00cefb5d4931bfa49483dde546413c9b40c4228220f27d7ecdf2'
       ]
df_ab_np=df_ab_np[df_ab_np['customer_id'].isin(id)]

In [37]:
df_ab_np=df_ab_np[df_ab_np['customer_id'].isin(amostra_aleatoria)]
df_ab_np.head()

,customer_id,is_target
7,2c2c1ec8e79d1b98e67aaefb39b477dd9c95bea6e498c8...,target
14,df4a3b2f7c0e259adb8e71e094405637810013013f4971...,control
18,e3c10f0f467527d2ac2ce784a0413db873c86c337f0c07...,control
24,680d1019ed0c75e464abd716eb2a337b7efb3789ee2a6a...,control
31,4c32c4bbabfe11d06b9b2285f46dbd57355b5404276ab8...,control


In [38]:
df_publico=merge_df(df_ab_np,df_consumer,['customer_id'],'inner')

In [39]:
clientes_mes = (
    df_publico.groupby(['active', 'is_target'], dropna=False)
              ['customer_id']
              .nunique()
              .reset_index(name='numero_clientes_distintos')
)
clientes_mes

,active,is_target,numero_clientes_distintos
0,False,control,100
1,False,target,123
2,True,control,44494
3,True,target,55247


In [40]:
df_publico = df_publico.dropna(subset=['active'])

In [41]:
clientes_mes = (
    df_publico.groupby(['active', 'is_target'], dropna=False)
              ['customer_id']
              .nunique()
              .reset_index(name='numero_clientes_distintos')
)
clientes_mes

,active,is_target,numero_clientes_distintos
0,False,control,100
1,False,target,123
2,True,control,44494
3,True,target,55247


Publico definido e todos os clientes marcados no teste a/b e existentes na base de clientes

Da base de ordens serao filtrados todos os clientes com orden nos meses de de dezembro e janeiro

In [ ]:
URL_ORDERS = "https://data-architect-test-source.s3-sa-east-1.amazonaws.com/order.json.gz"


COLUMNS_TO_DROP = [
    'cpf','customer_name','delivery_address_city','delivery_address_country',
   'delivery_address_district','delivery_address_external_id',
    'delivery_address_latitude','delivery_address_longitude',

    'delivery_address_zip_code','items',
    'merchant_latitude','merchant_longitude','merchant_timezone',
    'order_scheduled','order_scheduled_date'
]

customer_ids = df_ab["customer_id"].astype(str).unique()

df_orders = load_orders(
    url=URL_ORDERS,
    customer_ids=customer_ids,
    columns_to_drop=COLUMNS_TO_DROP
)

df_orders.head()


,customer_id,delivery_address_state,merchant_id,order_created_at,order_id,order_total_amount,origin_platform
0,7ba88a68bb2a3504c6bd37a707af57a0b8d6e110a551c7...,SP,a992a079a651e699d9149423761df2427c0e3af0a2a1b5...,2019-01-17T22:50:06.000Z,33e0612d62e5eb42aba15b58413137e441fbe906de2feb...,46.0,ANDROID
1,078acecdcf7fa89d356bfa349f14a8219db1ee161ce28a...,SP,5152f28ee0518b8803ccf0a4096eb2ff8b81e9491861c9...,2019-01-17T17:51:26.000Z,148c4353a2952f3fe7973547283265eb22b575fb712ed2...,104.5,ANDROID
2,0e38a3237b5946e8ab2367b4f1a3ae6e77f1e215bc760c...,SP,b6096419455c35d06105a5ef0d25c51f9dd40e1e99ac33...,2019-01-17T22:53:47.000Z,c37e495a91b498bb7b70a9e09ac115d0cdd443f152dc11...,35.0,IOS
3,cab1a004b7206d07910092c515a79834fea0a03d7d9054...,SP,082bfdcdf6ccdc343e3c4d25ee376b5b6ca7e96ad8b04e...,2019-01-17T23:56:53.000Z,b4df94142d21354611247da9ca94f870c09b93989b531a...,40.8,IOS
4,aa7edf5b166b8c843aec3b96dc561222888734f3879123...,ES,d7adb764bac29ccb77fb8f746ffbd531bf05ec30a7e130...,2019-01-17T23:40:53.000Z,4ff64b33b272c1886df21b63272220af6a82d1667dba70...,48.5,ANDROID


In [43]:
print(check_key_uniqueness(df_orders, ["customer_id","merchant_id","order_id"]))
print(check_key_uniqueness(df_orders, ["customer_id","merchant_id","order_id","order_created_at"]))

❌ Colunas ['customer_id', 'merchant_id', 'order_id'] possuem duplicações.
(False,                                                customer_id  \
1192509  c94d0872922e87c7f23669afcce3cc700d82ed75080a59...   
1192510  1e2af3429ee49b319a095258707980a7f1da2c2f1b6ca9...   
1192511  9a68122c178c32840eef9421530a375629d586ef9a7643...   
1192512  afdf6cab94f0e58e8a03940bb23243dc73263ad217205c...   
1192513  e9f79cc65b905e1e8cae0687605be386bc0275f6ec46ad...   
...                                                    ...   
3662316  648ae0e610811af0fccbe557b9a63a55c6e46adeeceb0b...   
3662317  5cab7f42316c5815d151d1fd0eebaecf9e6e53681257f0...   
3662318  1e91e110ba83f466ddbdb8ea448940e39e4e5ce16925e5...   
3662319  588becd71bc59b9a17ffcafe5823ce77777f308296f6f0...   
3662320  50862fb1670635160c98cd292768893dd65df05e5ae38e...   

        delivery_address_state  \
1192509                     BA   
1192510                     SP   
1192511                     SP   
1192512                     PR   
119

Add orders para a base de publico

In [44]:
df_publico_orders=merge_df(df_publico,df_orders,['customer_id'],'inner')

In [45]:
#df_publico_orders = df_publico_orders.drop('cpf', axis=1)
df_publico_orders['customer_id'].nunique()
df_publico_orders.shape
df_publico_orders.to_parquet(BASE_PATH / "silver" / "df_p_2.parquet", index=False)

Construcao de chave unica, e sumarizacoes visao cliente

In [46]:
#df_publico_orders[df_publico_orders['customer_id']=='fffe7b38b14ac2fca8906500fbecf87f5f8470179ed7ffa974892a3a15650604']

In [47]:
BASE_PATH

PosixPath('/Users/maceli/ifood_cs/dados')

In [48]:
#df_publico_orders = pd.read_parquet(BASE_PATH / "gold" / "df_publico_orders.parquet")

In [49]:
df_publico=process_orders_pandas(df_publico_orders)

In [50]:
df_publico


,customer_id,is_target,active,created_at,delivery_address_state,merchant_id,order_created_at,order_total_amount,origin_platform,order_created_month,unique_order_hash,total_amount_mes,ticket_medio,num_pedidos_mes,num_pedidos_hist,prev_order_time,diff_days
13260,000032b594523c3f8868edee4f1577b157e115cd01ab31...,control,True,2018-01-03T00:30:07.336Z,SP,f7047bb15db94feda0ff4cab08703324e806febc44b86b...,2018-12-15 23:41:56+00:00,88.8,ANDROID,12,860676046297272972,88.8,88.80,1,2,NaT,NaN
13259,000032b594523c3f8868edee4f1577b157e115cd01ab31...,control,True,2018-01-03T00:30:07.336Z,SP,f7047bb15db94feda0ff4cab08703324e806febc44b86b...,2019-01-14 23:41:56+00:00,88.8,ANDROID,1,4664756466039375947,88.8,88.80,1,2,2018-12-15 23:41:56+00:00,30.0
67978,0001004b6873a53fef60d11f4d2e4435ea27bed7e5fc38...,target,True,2018-01-28T16:23:12.535Z,SP,e9808433ad0812980fa03100e70b0f321ebc4bbad08f59...,2019-01-25 00:44:15+00:00,30.9,IOS,1,13108920244402146680,30.9,30.90,1,1,NaT,NaN
62548,000200d3759a5b4d0083d212773ae6e304cbe91ffddede...,target,True,2018-01-08T01:10:00.719Z,SP,ad65673d64a12f6f590e3654c798f02cd3de2ee6e1450a...,2018-12-06 01:28:49+00:00,47.6,ANDROID,12,12782966171788076350,47.6,47.60,1,2,NaT,NaN
62547,000200d3759a5b4d0083d212773ae6e304cbe91ffddede...,target,True,2018-01-08T01:10:00.719Z,SP,ad65673d64a12f6f590e3654c798f02cd3de2ee6e1450a...,2019-01-05 01:28:49+00:00,47.6,ANDROID,1,4954256155446447672,47.6,47.60,1,2,2018-12-06 01:28:49+00:00,30.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120417,fffebe8f36625fbc708361eeaa301fee40b849e2bf4666...,target,True,2018-01-03T15:34:50.747Z,RN,a63b22b8318666bcdd671fef92c281bbec5c0281de4af7...,2019-01-15 23:33:34+00:00,34.6,IOS,1,6882048171816698458,96.3,48.15,2,4,2019-01-11 14:43:29+00:00,4.0
451694,fffef917711775d1ba63ec1d8054f9705177edef950699...,control,True,2018-04-05T13:16:05.922Z,SP,91955dd1b84ae9fc5d8df383524b74fc809ea568b61f09...,2018-12-14 00:20:43+00:00,23.8,IOS,12,10741791939650141914,23.8,23.80,1,2,NaT,NaN
451693,fffef917711775d1ba63ec1d8054f9705177edef950699...,control,True,2018-04-05T13:16:05.922Z,SP,91955dd1b84ae9fc5d8df383524b74fc809ea568b61f09...,2019-01-13 00:20:43+00:00,23.8,IOS,1,299850713772058177,23.8,23.80,1,2,2018-12-14 00:20:43+00:00,30.0
20468,ffff43a3295b205e9024717bd803b218f49aa87c8e2b90...,control,True,2018-01-08T18:47:14.979Z,RJ,e204d9d5826c150c86e17ed2c5c66d7e3169ed1de4d7de...,2018-12-19 23:26:53+00:00,59.0,ANDROID,12,3855802819150875312,59.0,59.00,1,2,NaT,NaN


In [51]:
#save_parquet(df_publico, "silver", "df_publico.parquet")

In [52]:
#df_publico_orders[df_publico_orders['customer_id']=='fffe7bx38b14ac2fca8906500fbecf87f5f8470179ed7ffa974892a3a15650604']

Uma linha por cliente, com as variaveis necessarias

In [53]:
#df_publico[df_publico['customer_id']=='fffe7b38b14ac2fca8906500fbecf87f5f8470179ed7ffa974892a3a15650604']

In [54]:
#df_publico['customer_id'].unique()

In [55]:
df_pub_un = df_publico[["customer_id","is_target", "order_created_month", "num_pedidos_mes", "num_pedidos_hist",'total_amount_mes','ticket_medio']].drop_duplicates().reset_index(drop=True)
df_pub_un['pedidos_sum'] = np.where(
        df_pub_un['num_pedidos_mes'] > 10, '10+',
        df_pub_un['num_pedidos_mes'].astype(str)
    )
df_pub_un.head()

,customer_id,is_target,order_created_month,num_pedidos_mes,num_pedidos_hist,total_amount_mes,ticket_medio,pedidos_sum
0,000032b594523c3f8868edee4f1577b157e115cd01ab31...,control,12,1,2,88.8,88.8,1
1,000032b594523c3f8868edee4f1577b157e115cd01ab31...,control,1,1,2,88.8,88.8,1
2,0001004b6873a53fef60d11f4d2e4435ea27bed7e5fc38...,target,1,1,1,30.9,30.9,1
3,000200d3759a5b4d0083d212773ae6e304cbe91ffddede...,target,12,1,2,47.6,47.6,1
4,000200d3759a5b4d0083d212773ae6e304cbe91ffddede...,target,1,1,2,47.6,47.6,1


Salvar base visao cliente em gold layer

In [56]:
df_pub_un.to_parquet(BASE_PATH / "gold" / "df_pub_un.parquet", index=False)

In [57]:
df_stats_mes = df_pub_un.groupby(['pedidos_sum', 'order_created_month']).agg(
    total_clientes=('customer_id', 'nunique')
)

df_stats_mes['pct_total_mes'] = (
    df_stats_mes['total_clientes'] /
    df_stats_mes.groupby('order_created_month')['total_clientes'].transform('sum') * 100
).round(2)

df_stats_mes.head(20)


total_clientes  pct_total_mes
pedidos_sum order_created_month                               
1           1                             46558          46.57
            12                            37161          54.11
10          1                               968           0.97
            12                              278           0.40
10+         1                              4271           4.27
            12                              886           1.29
2           1                             18850          18.86
            12                            13817          20.12
3           1                             10232          10.24
            12                             6974          10.15
4           1                              6494           6.50
            12                             3799           5.53
5           1                              4400           4.40
            12                             2251           3.28
6           1                              3088           3.09
            12                             1451           2.11
7           1                              2216           2.22
            12                              963           1.40
8           1                              1643           1.64
            12                              654           0.95

In [58]:
df_pub_hist = df_publico[["customer_id","is_target",  "num_pedidos_hist"]].drop_duplicates().reset_index(drop=True)

In [59]:
df_stats_hist = df_pub_hist.groupby(['num_pedidos_hist']).agg(
    total_clientes=('customer_id', 'nunique')  
).round(2)

df_stats_hist['pct_total'] = (df_stats_hist['total_clientes'] / df_stats_hist['total_clientes'].sum() * 100).round(2)

df_stats_hist.head(20)

,total_clientes,pct_total
num_pedidos_hist,,
1,22502,22.51
2,29672,29.68
3,8885,8.89
4,10084,10.09
5,4850,4.85
6,5056,5.06
7,2906,2.91
8,2844,2.85
9,1972,1.97
